# IT2011 – Artificial Intelligence and Machine Learning
## Progress Review I – Combined Data Preprocessing Pipeline

**Group ID:** MLB-B14G1-02
**Dataset:** Brain Tumor Classification (MRI) — Kaggle / Sartaj Bhuvaji
**Source:** https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri
**Classes:** `glioma_tumor`, `meningioma_tumor`, `no_tumor`, `pituitary_tumor`

| Student ID | Member | Assigned Technique | Feeds Into |
|---|---|---|---|
| IT25200148 | Normalization | Data cleaning, resizing, pixel scaling | Sections 2, 5, 6 |
| IT25101873 | Duplicate Detection & Splitting | SHA-256 exact-duplicate detection, leakage-safe train/val/test split | Sections 1, 3, 4 |
| IT25100781 | Encoding Categorical Variables | Label/One-Hot encoding of target classes | Sections 7, 11 |
| IT25101560 | Outlier Removal, Normalization, Encoding & Transfer Learning | Outlier removal, PCA, class weighting, VGG16 fine-tuning | Sections 2, 8, 9, 10, 11 |

This notebook does **not** just paste the four individual notebooks one after another. Each
member's code has been re-sequenced into one logically-ordered pipeline, the places where two
members solved the same problem differently (two corrupt/outlier checks, two normalization
styles, two label-encoding APIs) have been reconciled into one shared implementation, and a
single set of DataFrames/arrays is threaded through every stage so the output of one member's
stage is exactly the input the next member's stage expects.

### Combined Pipeline Flow

```
[IT25101873]                    [IT25200148 + IT25101560]          [IT25101873]
1. Build master DataFrame  ───▶ 2. Clean: corrupt-file check  ───▶ 3. SHA-256 exact-duplicate
   from Training/ + Testing/       (PIL.verify) + outlier check       detection + cross-label
                                    (blank / tiny, cv2)                conflict guard
        │
        ▼
[IT25101873]
4. Duplicate-aware stratified split
   (80% train-pool / 20% test, then 85%/15% internal train/val)
   — duplicate groups are never split across sets (zero leakage)
        │
        ▼
[IT25200148 + IT25101560]                    [IT25100781 + IT25101560]
5. Load images → resize 224×224 ───────────▶ 7. Encode target labels
   → RGB → Min-Max [0,1] scaling                (LabelEncoder + OneHotEncoder,
   (+ ImageNet standardization demo)              cross-checked vs LabelBinarizer)
        │                                              │
        ▼                                              ▼
[IT25101560]                                  [ALL 4 MEMBERS]
6. Class-weight computation ◀──────────────── 8. Consolidated EDA
   (train split only, imbalance-aware)            (distribution, duplicates, pixel
                                                    stats, correlation, sparsity, PCA)
        │
        ▼
[IT25101560]                                  [IT25100781 + IT25101560]
9. Augmentation generators ─────────────────▶ 10. Model integration check
   (flow_from_dataframe, train split only)         (ANN baseline + VGG16 head)
```

Every arrow above is a real variable handed from one code cell to the next: the split
DataFrames (`fit_train_df`, `val_df`, `test_df`) built in Section 4 are what Section 5 loads
images from, the arrays Section 5/6 produce are what Section 7's encoder is fit against, and
the class weights from Section 6 feed both models in Section 11.

In [ ]:
# =====================================================================
# GLOBAL SETUP & CONFIGURATION
# Adapted from IT25101873's configuration block (clean path handling,
# fixed seeds, reproducible output folders) — this becomes the shared
# configuration every other member's stage reads from below.
# =====================================================================
from pathlib import Path
import hashlib
import random
import warnings

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split  # fallback only, see Section 4

warnings.filterwarnings("ignore")

# --- Try to import TensorFlow/Keras. Sections 9-11 need it; Sections 1-8 do not. ---
# This keeps the data-preparation half of the pipeline runnable even on a machine
# that hasn't installed TensorFlow yet (e.g. for a quick review of the preprocessing).
try:
    import tensorflow as tf
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    from tensorflow.keras.applications import VGG16
    from tensorflow.keras.models import Sequential, Model
    from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
    TENSORFLOW_AVAILABLE = True
    print(f"TensorFlow {tf.__version__} detected — full pipeline (Sections 1-11) will run.")
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print("TensorFlow not found — Sections 1-8 (data preparation) will still run in full.\n"
          "Install `tensorflow` to also run Sections 9-11 (augmentation + model integration).")

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- Class configuration (shared across every member's notebook) ---
CLASS_NAMES = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]
DISPLAY_NAMES = {
    "glioma_tumor": "Glioma Tumor",
    "meningioma_tumor": "Meningioma Tumor",
    "no_tumor": "No Tumor",
    "pituitary_tumor": "Pituitary Tumor",
}
IMG_SIZE = 224            # target spatial resolution used by every downstream model
TEST_FRACTION = 0.20      # final untouched hold-out
VAL_FRACTION_OF_TRAIN = 0.15   # internal validation slice of the remaining 80%

# --- Paths & output folders ---
PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "results" / "outputs"
EDA_DIR = PROJECT_ROOT / "results" / "eda_visualizations"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EDA_DIR.mkdir(parents=True, exist_ok=True)

# The dataset is searched for under a few common layouts so this notebook runs
# unmodified on any team member's machine. Each candidate must contain BOTH a
# "Training" and a "Testing" subfolder with the four class folders inside.
_candidate_roots = [
    PROJECT_ROOT / "data" / "brain_tumor_mri",
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT,
    Path(r"C:\\Users\\User\\Documents"),
]

def _looks_like_dataset_root(root: Path) -> bool:
    return (root / "Training").exists() and (root / "Testing").exists()

DATA_DIR = next((r for r in _candidate_roots if _looks_like_dataset_root(r)), None)

if DATA_DIR is None:
    print("Real dataset not found under any candidate path.")
    print("Candidates checked:", [str(p) for p in _candidate_roots])
    print("-> Falling back to a small SYNTHETIC demo dataset so the whole pipeline is "
          "still runnable end-to-end. Point DATA_DIR at your real Training/ and "
          "Testing/ folders (each containing the 4 class subfolders) to use real data.")
else:
    print("Dataset located at:", DATA_DIR)


## Section 1 — Data Ingestion: Building the Master DataFrame
**Owner: IT25101873**

Instead of treating the Kaggle `Training/` and `Testing/` folders as fixed splits (as the raw
download provides them), IT25101873's approach pools every image from both folders into one
master DataFrame and re-splits it later (Section 4) using a leakage-aware, duplicate-aware
method. This is more robust than trusting the original folder split, and it is the design this
combined pipeline keeps as the backbone that every other member's stage builds on.

If no real dataset is found (see the setup cell above), a tiny synthetic dataset is generated
here instead purely so the rest of the notebook can be demonstrated end-to-end.

In [ ]:
# =====================================================================
# SECTION 1: BUILD MASTER IMAGE DATAFRAME — IT25101873
# =====================================================================

def _make_synthetic_dataset(root: Path, n_train=30, n_test=8):
    '''Generates a small placeholder dataset with the correct folder layout so the
    notebook can be run end-to-end even without the real Kaggle download. Not used
    when a real dataset is found.'''
    rng = np.random.default_rng(SEED)
    for split, n in [("Training", n_train), ("Testing", n_test)]:
        for cls in CLASS_NAMES:
            d = root / split / cls
            d.mkdir(parents=True, exist_ok=True)
            for i in range(n):
                arr = (rng.random((256, 256, 3)) * 255).astype(np.uint8)
                Image.fromarray(arr).save(d / f"{cls}_{split}_{i:03d}.jpg", quality=90)
    return root

if DATA_DIR is None:
    DATA_DIR = _make_synthetic_dataset(PROJECT_ROOT / "demo_data" / "brain_tumor_mri")
    print("Synthetic demo dataset created at:", DATA_DIR)


def build_image_dataframe(data_dir: Path) -> pd.DataFrame:
    '''Walks Training/ and Testing/ under data_dir and returns one row per image
    with its path, filename, class label and which original Kaggle split it came
    from (kept only as metadata — the real split is decided in Section 4).'''
    rows = []
    for original_split in ["Training", "Testing"]:
        split_dir = data_dir / original_split
        if not split_dir.exists():
            raise FileNotFoundError(f"Missing directory: {split_dir}")
        for label in CLASS_NAMES:
            class_dir = split_dir / label
            if not class_dir.exists():
                raise FileNotFoundError(f"Missing class directory: {class_dir}")
            for path in sorted(class_dir.glob("*")):
                if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
                    rows.append({
                        "path": str(path.resolve()),
                        "filename": path.name,
                        "label": label,
                        "original_split": original_split,
                    })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No image files were found.")
    return df


raw_df = build_image_dataframe(DATA_DIR)
print("Total raw images indexed:", len(raw_df))
print(raw_df.groupby(["original_split", "label"]).size().unstack(fill_value=0)
      .reindex(columns=CLASS_NAMES))
raw_df.head()


## Section 2 — Data Cleaning: Corrupt-File & Outlier Detection
**Owners: IT25200148 (corrupt/missing-file check) + IT25101560 (outlier removal)**

Both members independently wrote a validity check before doing anything else with the images —
IT25200148 used `PIL.Image.verify()` to catch corrupt/truncated files, and IT25101560 used
`cv2.imread` returning `None` plus a blank/tiny-image check to catch outliers. Rather than pick
one, this section runs **both** checks together in a single pass so nothing either member would
have caught slips through, and tags every file with a status instead of silently dropping it.

In [ ]:
# =====================================================================
# SECTION 2: DATA CLEANING — CORRUPT & OUTLIER DETECTION
# Merge of IT25200148's PIL.verify() integrity check and IT25101560's
# cv2-based unreadable / blank / undersized image check.
# =====================================================================

MIN_VALID_DIM = 50   # IT25101560's threshold for a "tiny" outlier image

def validate_image(path_str: str, min_dim: int = MIN_VALID_DIM):
    '''Returns (status, reason). status is one of:
    'valid', 'corrupt' (unreadable / truncated), 'outlier_blank', 'outlier_tiny'.'''
    # --- IT25200148: structural integrity check ---
    try:
        with Image.open(path_str) as img:
            img.verify()
    except Exception as e:
        return "corrupt", f"PIL verify failed: {e}"

    # --- IT25101560: readability + outlier check (re-open, verify() consumes the handle) ---
    arr = cv2.imread(path_str, cv2.IMREAD_COLOR)
    if arr is None:
        return "corrupt", "cv2.imread returned None"
    if arr.shape[0] < min_dim or arr.shape[1] < min_dim:
        return "outlier_tiny", f"shape={arr.shape}"
    if np.sum(arr) == 0:
        return "outlier_blank", "all-zero pixel values"

    return "valid", None


print("Running combined corrupt-file + outlier check on", len(raw_df), "images...")
_results = raw_df["path"].apply(validate_image)
raw_df["status"] = [r[0] for r in _results]
raw_df["issue_reason"] = [r[1] for r in _results]

print("\n--- Cleaning summary ---")
print(raw_df["status"].value_counts())

issues_df = raw_df[raw_df["status"] != "valid"]
if len(issues_df):
    print("\nFlagged files (excluded from the pipeline):")
    print(issues_df[["filename", "label", "status", "issue_reason"]].to_string(index=False))

clean_df = raw_df[raw_df["status"] == "valid"].drop(columns=["status", "issue_reason"]).reset_index(drop=True)
print(f"\nSTEP COMPLETE: {len(clean_df)} / {len(raw_df)} images passed cleaning "
      f"({len(raw_df) - len(clean_df)} removed).")


**Interpretation:** Corrupt or unreadable files (truncated downloads, zero-byte files)
and outliers (blank slices, abnormally small images) are excluded *before* hashing, splitting,
or any statistics are computed — so later duplicate-detection and class-balance numbers in
Section 4/8 are not skewed by files that were never going to reach a model anyway.

## Section 3 — Exact Duplicate Detection
**Owner: IT25101873**

Computes a SHA-256 hash of every cleaned image's bytes. Two files with the same hash are
byte-identical. This section (a) guards against the same image ever being assigned two
different class labels, and (b) groups identical files together so Section 4 can keep every
copy of a duplicate on the same side of the train/val/test split (preventing data leakage).

In [ ]:
# =====================================================================
# SECTION 3: EXACT DUPLICATE DETECTION (SHA-256) — IT25101873
# =====================================================================

def sha256_file(path_str: str, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path_str, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


print("Hashing", len(clean_df), "cleaned images...")
clean_df["sha256"] = clean_df["path"].apply(sha256_file)

# --- Safety check: an identical image must never carry two different labels ---
cross_label_hashes = clean_df.groupby("sha256")["label"].nunique()
cross_label_hashes = cross_label_hashes[cross_label_hashes > 1]
print("Exact-duplicate hashes assigned to multiple labels:", len(cross_label_hashes))
if len(cross_label_hashes) > 0:
    display_cols = ["path", "label", "original_split", "sha256"]
    print(clean_df[clean_df["sha256"].isin(cross_label_hashes.index)]
          .sort_values("sha256")[display_cols])
    raise RuntimeError("Conflicting labels found for identical images — fix the dataset before continuing.")

# --- Duplicate groups within each class (informational, not an error) ---
dup_groups = (clean_df.groupby(["label", "sha256"]).size().reset_index(name="copies"))
duplicate_groups = dup_groups[dup_groups["copies"] > 1]
print("Duplicate groups (2+ identical files):", len(duplicate_groups))
print("Extra files beyond one copy per group:", int((duplicate_groups["copies"] - 1).sum()) if len(duplicate_groups) else 0)

duplicate_summary = (
    duplicate_groups.groupby("label")
    .agg(duplicate_groups=("copies", "size"), images_in_duplicate_groups=("copies", "sum"))
    .reindex(CLASS_NAMES).fillna(0)
)
print(duplicate_summary)

plt.figure(figsize=(8, 4))
plt.bar([DISPLAY_NAMES[c] for c in CLASS_NAMES], duplicate_summary["duplicate_groups"].values, color="#3690c0")
plt.title("Exact Duplicate Groups by Class")
plt.ylabel("Number of Duplicate Groups")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(EDA_DIR / "03_duplicate_groups_by_class.png", dpi=150)
plt.show()


## Section 4 — Duplicate-Aware Stratified Train / Validation / Test Split
**Owner: IT25101873**

A plain `train_test_split` could put one copy of a duplicated image in the training set and its
identical twin in the test set — the model would then be evaluated on an image it has
effectively already seen, inflating test accuracy. IT25101873's split solves this with two
pieces:

1. `largest_remainder_targets` — decides exactly how many *images* each class should contribute
   to the test set (or validation set) while keeping the overall fraction (e.g. 20%) exact after
   rounding.
2. `choose_groups_exactly` — a small subset-sum search that selects whole duplicate-hash groups
   (never splitting one) whose sizes add up to precisely that target.

The result: an 80% train-pool / 20% test split, then the 80% pool is split again 85% / 15% into
the actual training set and an internal validation set — with **zero** duplicate images crossing
any boundary, verified explicitly below.

In [ ]:
# =====================================================================
# SECTION 4: DUPLICATE-AWARE STRATIFIED SPLIT — IT25101873
# =====================================================================

def largest_remainder_targets(class_counts: pd.Series, fraction: float) -> pd.Series:
    '''How many images per class should land in the target subset, using the
    largest-remainder method so per-class targets sum exactly to round(total*fraction).'''
    exact = class_counts.astype(float) * fraction
    base = np.floor(exact).astype(int)
    target_total = int(round(class_counts.sum() * fraction))
    remaining = target_total - int(base.sum())
    fractional_parts = (exact - base).sort_values(ascending=False)
    result = base.copy()
    for label in fractional_parts.index[:remaining]:
        result.loc[label] += 1
    return result.astype(int)


def choose_groups_exactly(group_sizes: pd.Series, target: int, seed: int):
    '''Subset-sum search: picks a set of duplicate-hash groups (each an indivisible
    block of `copies` identical images) whose sizes sum to exactly `target`.'''
    rng = np.random.default_rng(seed)
    ids = group_sizes.index.to_numpy(copy=True)
    rng.shuffle(ids)
    dp = {0: ()}
    for group_id in ids:
        size = int(group_sizes.loc[group_id])
        for current_sum, chosen in list(dp.items()):
            new_sum = current_sum + size
            if new_sum > target or new_sum in dp:
                continue
            dp[new_sum] = chosen + (group_id,)
            if new_sum == target:
                return set(dp[new_sum])
    raise RuntimeError(f"Could not build an exact grouped split for target={target}. "
                        "Try a different seed or inspect unusually large duplicate groups.")


def make_duplicate_aware_stratified_split(data: pd.DataFrame, fraction: float, seed: int):
    class_counts = data["label"].value_counts().reindex(CLASS_NAMES)
    targets = largest_remainder_targets(class_counts, fraction)

    held_out_indices = []
    for class_number, label in enumerate(CLASS_NAMES):
        class_df = data[data["label"] == label]
        group_sizes = class_df.groupby("sha256").size().sort_index()
        chosen_hashes = choose_groups_exactly(group_sizes, int(targets.loc[label]), seed=seed + class_number)
        idxs = class_df[class_df["sha256"].isin(chosen_hashes)].index.tolist()
        assert len(idxs) == int(targets.loc[label]), f"{label}: split target mismatch"
        held_out_indices.extend(idxs)

    held_out_df = data.loc[held_out_indices].sample(frac=1, random_state=seed).reset_index(drop=True)
    remainder_df = data.drop(index=held_out_indices).sample(frac=1, random_state=seed).reset_index(drop=True)
    return remainder_df, held_out_df, targets


# --- 80% train-pool / 20% final test ---
train_pool_df, test_df, _ = make_duplicate_aware_stratified_split(clean_df, TEST_FRACTION, SEED)

# --- Internal 85% / 15% split of the train-pool into fit-train / validation ---
fit_train_df, val_df, _ = make_duplicate_aware_stratified_split(train_pool_df, VAL_FRACTION_OF_TRAIN, SEED + 100)

split_table = pd.DataFrame({
    "Total": clean_df["label"].value_counts().reindex(CLASS_NAMES),
    "Fit train": fit_train_df["label"].value_counts().reindex(CLASS_NAMES),
    "Validation": val_df["label"].value_counts().reindex(CLASS_NAMES),
    "Test": test_df["label"].value_counts().reindex(CLASS_NAMES),
})
print(split_table.rename(index=DISPLAY_NAMES))
print(f"\nTotals -> fit_train: {len(fit_train_df)}  val: {len(val_df)}  test: {len(test_df)}  "
      f"(sum={len(fit_train_df)+len(val_df)+len(test_df)}, expected={len(clean_df)})")

# --- Leakage verification: no SHA-256 hash appears in more than one split ---
h_train, h_val, h_test = set(fit_train_df.sha256), set(val_df.sha256), set(test_df.sha256)
assert not (h_train & h_val), "Leakage between train and validation!"
assert not (h_train & h_test), "Leakage between train and test!"
assert not (h_val & h_test), "Leakage between validation and test!"
assert len(fit_train_df) + len(val_df) + len(test_df) == len(clean_df)
print("Leakage check passed: no duplicate image crosses a split boundary.")

# --- EDA: class distribution across the final splits ---
plot_df = split_table.copy()
plot_df.index = [DISPLAY_NAMES[c] for c in plot_df.index]
ax = plot_df.plot(kind="bar", figsize=(10, 5))
ax.set_title("Class Distribution: Total vs Fit-Train / Validation / Test")
ax.set_ylabel("Number of Images")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(EDA_DIR / "04_split_class_distribution.png", dpi=150)
plt.show()

for name, d in [("train_80_percent", train_pool_df), ("test_20_percent", test_df),
                ("fit_train", fit_train_df), ("validation", val_df)]:
    d.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
print("Split manifests saved to", OUTPUT_DIR)


## Section 5 — Image Loading, Resizing & Colour Standardization
**Owners: IT25200148 (resizing/dimension inspection) + IT25101560 (cv2 load pipeline)**

Both members resized to 224×224 (the input size every downstream model in this project —
the ANN baseline and VGG16 — expects) and both handled the OpenCV BGR→RGB channel-order
issue. This section merges the two into one loader function and applies it separately to each
of the three splits produced in Section 4, so no image is read into memory before its split is
decided.

In [ ]:
# =====================================================================
# SECTION 5: IMAGE LOADING, RESIZING & RGB STANDARDIZATION
# Merge of IT25200148's PIL-based resize inspection and IT25101560's
# cv2-based batch loader.
# =====================================================================

def load_and_resize_images(df: pd.DataFrame, img_size: int = IMG_SIZE) -> np.ndarray:
    '''Loads every image in df['path'], resizes with bilinear interpolation to
    (img_size, img_size), converts OpenCV's default BGR to RGB, and returns a
    uint8 array of shape (N, img_size, img_size, 3) — pixel values still in [0, 255].
    Normalization is a separate, explicit step (Section 6).'''
    X = np.zeros((len(df), img_size, img_size, 3), dtype=np.uint8)
    for i, path in enumerate(df["path"].values):
        img = cv2.imread(path, cv2.IMREAD_COLOR)               # BGR, already validated in Section 2
        img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        X[i] = img
    return X


# --- Dimension inspection on one sample, mirroring IT25200148's diagnostic step ---
_sample_path = fit_train_df["path"].iloc[0]
with Image.open(_sample_path) as _sample:
    print(f"Original sample dimensions : {_sample.size[0]}x{_sample.size[1]}px, mode={_sample.mode}")
print(f"Target dimensions after resize: {IMG_SIZE}x{IMG_SIZE}px, mode=RGB")

print("\nLoading and resizing each split (this may take a while on the full dataset)...")
X_train_raw = load_and_resize_images(fit_train_df)
X_val_raw = load_and_resize_images(val_df)
X_test_raw = load_and_resize_images(test_df)
print("X_train_raw:", X_train_raw.shape, "| X_val_raw:", X_val_raw.shape, "| X_test_raw:", X_test_raw.shape)


## Section 6 — Pixel Normalization
**Owner: IT25200148**

Two normalization styles were used across the group's notebooks: plain Min-Max scaling to
`[0, 1]` (used to feed the ANN baseline and as the base of VGG16's `ImageDataGenerator`
rescaling), and ImageNet mean/std standardization (a common transfer-learning practice, since
VGG16's original weights were trained on ImageNet-normalized inputs). Both are kept here: Min-Max
scaling is applied to the full dataset for every downstream stage, and ImageNet standardization
is demonstrated (and available as `normalize_imagenet`) for the VGG16 branch in Section 11.

In [ ]:
# =====================================================================
# SECTION 6: PIXEL NORMALIZATION — IT25200148
# =====================================================================

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def normalize_min_max(X_uint8: np.ndarray) -> np.ndarray:
    '''Scales uint8 pixel values from [0, 255] to [0.0, 1.0].'''
    return X_uint8.astype(np.float32) / 255.0

def normalize_imagenet(X_scaled_0_1: np.ndarray) -> np.ndarray:
    '''Zero-centers an already-[0,1]-scaled batch using ImageNet channel statistics —
    the standardization VGG16's pretrained weights expect.'''
    return (X_scaled_0_1 - IMAGENET_MEAN) / IMAGENET_STD


X_train = normalize_min_max(X_train_raw)
X_val = normalize_min_max(X_val_raw)
X_test = normalize_min_max(X_test_raw)
print(f"Min-Max scaled range -> train: [{X_train.min():.3f}, {X_train.max():.3f}]")

# --- Demonstration on a single sample, mirroring IT25200148's before/after histogram ---
_raw_sample = X_train_raw[0].astype(np.float32)
_scaled_sample = X_train[0]
_imagenet_sample = normalize_imagenet(_scaled_sample)

print(f"1. Raw pixel range        : min={_raw_sample.min():.1f}, max={_raw_sample.max():.1f}")
print(f"2. Min-Max scaled range   : min={_scaled_sample.min():.4f}, max={_scaled_sample.max():.4f}")
print(f"3. ImageNet standardized  : min={_imagenet_sample.min():.4f}, max={_imagenet_sample.max():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(_raw_sample.ravel(), bins=50, color="gray", alpha=0.8)
axes[0].set_title("Raw Pixel Intensity [0, 255]")
axes[0].set_xlabel("Pixel Value"); axes[0].set_ylabel("Frequency")

axes[1].hist(_scaled_sample.ravel(), bins=50, color="royalblue", alpha=0.8)
axes[1].set_title("Min-Max Normalized [0, 1]")
axes[1].set_xlabel("Pixel Value"); axes[1].set_ylabel("Frequency")
plt.tight_layout()
plt.savefig(EDA_DIR / "06_pixel_normalization.png", dpi=150)
plt.show()


**Interpretation:** Min-Max scaling puts every image on the same `[0, 1]` scale regardless
of the original MRI scanner's brightness/contrast settings, which is required before computing
any distance-based statistic (PCA in Section 8) or feeding a neural network — large unscaled
pixel values would otherwise dominate early gradient updates.

## Section 7 — Categorical Label Encoding
**Owner: IT25100781** (cross-checked against IT25101560's `LabelBinarizer` usage)

The raw labels are class-name strings (`glioma_tumor`, ...). IT25100781's approach —
`LabelEncoder` to get integer indices, then `OneHotEncoder` to get orthogonal binary vectors —
is used here as the canonical implementation. IT25101560 achieved an equivalent result in the
VGG16 notebook with a single `LabelBinarizer` call; both encoders are fit **once**, on the fixed
`CLASS_NAMES` list (not on any one split), so the column order is guaranteed identical across
`y_train`, `y_val` and `y_test` even if a class happened to be under-represented in a split.

In [ ]:
# =====================================================================
# SECTION 7: CATEGORICAL LABEL ENCODING — IT25100781
# =====================================================================

# Step A: Integer label encoding, fit on the fixed class list (not on a split) so
# every split maps class names to the exact same integer indices.
label_encoder = LabelEncoder().fit(CLASS_NAMES)
print("Label -> integer mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

y_train_int = label_encoder.transform(fit_train_df["label"])
y_val_int = label_encoder.transform(val_df["label"])
y_test_int = label_encoder.transform(test_df["label"])

# Step B: One-hot encoding, categories fixed to all 4 classes so the output always
# has 4 columns even if a split is missing a class.
onehot_encoder = OneHotEncoder(sparse_output=False, categories=[np.arange(len(CLASS_NAMES))])
onehot_encoder.fit(np.arange(len(CLASS_NAMES)).reshape(-1, 1))

y_train = onehot_encoder.transform(y_train_int.reshape(-1, 1))
y_val = onehot_encoder.transform(y_val_int.reshape(-1, 1))
y_test = onehot_encoder.transform(y_test_int.reshape(-1, 1))

# Sanity checks: exactly one active unit per row (matches IT25100781's "EDA 3" check)
assert (y_train.sum(axis=1) == 1).all()
assert (y_val.sum(axis=1) == 1).all()
assert (y_test.sum(axis=1) == 1).all()
print("y_train shape:", y_train.shape, "| y_val shape:", y_val.shape, "| y_test shape:", y_test.shape)
print("Each row sums to 1.0 (one-hot property verified) across all three splits.")

# Equivalence note: `sklearn.preprocessing.LabelBinarizer().fit_transform(labels)`,
# as used in IT25101560's notebook, produces the same N x 4 binary matrix as the
# LabelEncoder + OneHotEncoder combination above for this 4-class problem.


## Section 8 — Class Imbalance Handling: Class Weights
**Owner: IT25101560**

The dataset is imbalanced (`no_tumor` has noticeably fewer images than the three tumor
classes). Rather than duplicating or discarding images, IT25101560's notebook computes balanced
class weights with `sklearn.utils.class_weight.compute_class_weight` and passes them to
`model.fit(..., class_weight=...)`. Those weights are computed **only from the training split**
(`y_train_int`) — never from validation or test — so no information about the held-out sets
leaks into training.

In [ ]:
# =====================================================================
# SECTION 8: CLASS WEIGHT COMPUTATION — IT25101560
# =====================================================================

classes_present = np.unique(y_train_int)
weights = compute_class_weight(class_weight="balanced", classes=classes_present, y=y_train_int)
class_weights = dict(zip(classes_present.tolist(), weights.tolist()))

print("Class weights (train split only):")
for idx, w in class_weights.items():
    print(f"  {label_encoder.classes_[idx]:<18}: {w:.4f}")

plt.figure(figsize=(7, 4))
plt.bar([DISPLAY_NAMES[label_encoder.classes_[i]] for i in class_weights],
        list(class_weights.values()), color="#02818a")
plt.title("Balanced Class Weights (Training Split)")
plt.ylabel("Weight")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(EDA_DIR / "08_class_weights.png", dpi=150)
plt.show()


## Section 9 — Consolidated Exploratory Data Analysis
**Owners: all four members**

Each member's notebook included its own EDA. Rather than repeat six separate figures, this
section reproduces the distinct, non-overlapping visualizations from all four notebooks as one
consolidated panel: class balance, pixel statistics, encoding correctness, and structure in the
image space via PCA.

In [ ]:
# =====================================================================
# SECTION 9: CONSOLIDATED EDA — ALL MEMBERS
# =====================================================================
fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# --- 1. Raw class frequency (IT25100781 / IT25200148) ---
sns.countplot(x=clean_df["label"], ax=axes[0, 0], hue=clean_df["label"], palette="crest", legend=False)
axes[0, 0].set_title("1. Class Frequency (post-cleaning)", fontweight="bold")
axes[0, 0].set_xlabel("Diagnosis Class"); axes[0, 0].set_ylabel("Sample Count")
axes[0, 0].tick_params(axis="x", rotation=20)

# --- 2. One-hot inter-class correlation heatmap (IT25100781) ---
corr = pd.DataFrame(y_train, columns=CLASS_NAMES).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", ax=axes[0, 1], cbar=True)
axes[0, 1].set_title("2. One-Hot Inter-Class Correlation", fontweight="bold")
axes[0, 1].tick_params(axis="x", rotation=25)

# --- 3. Encoded vector sparsity, first 40 training samples (IT25100781) ---
sns.heatmap(y_train[:40], cmap="Blues", cbar=False, ax=axes[0, 2], linewidths=0.5)
axes[0, 2].set_title("3. One-Hot Vector Sparsity (first 40 samples)", fontweight="bold")
axes[0, 2].set_xlabel("Target Neuron (0-3)"); axes[0, 2].set_ylabel("Sample Index")

# --- 4. Normalized pixel intensity distribution (IT25200148 / IT25101560) ---
axes[1, 0].hist(X_train.ravel(), bins=50, color="royalblue", alpha=0.8)
axes[1, 0].set_title("4. Normalized Pixel Intensity [0,1] (train)", fontweight="bold")
axes[1, 0].set_xlabel("Pixel Value"); axes[1, 0].set_ylabel("Frequency")

# --- 5. Class-wise mean intensity boxplot (IT25200148) ---
_gray_means = {cls: [] for cls in CLASS_NAMES}
for i, lbl in enumerate(fit_train_df["label"].values):
    _gray_means[lbl].append(X_train[i].mean())
box = axes[1, 1].boxplot([_gray_means[c] for c in CLASS_NAMES], patch_artist=True,
                          labels=[DISPLAY_NAMES[c] for c in CLASS_NAMES])
for patch, color in zip(box["boxes"], ["#2b5c8f", "#3690c0", "#67a9cf", "#02818a"]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1, 1].set_title("5. Mean Intensity by Class (train)", fontweight="bold")
axes[1, 1].set_ylabel("Mean Normalized Intensity")
axes[1, 1].tick_params(axis="x", rotation=15)

# --- 6. PCA of flattened normalized images (IT25101560) ---
X_flat = X_train.reshape(len(X_train), -1)
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_flat)
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=fit_train_df["label"].values,
                 palette="Set1", s=50, alpha=0.7, ax=axes[1, 2])
axes[1, 2].set_title("6. PCA of Training Images (2 components)", fontweight="bold")
axes[1, 2].set_xlabel("PC 1"); axes[1, 2].set_ylabel("PC 2")

plt.tight_layout()
plt.savefig(EDA_DIR / "09_consolidated_eda.png", dpi=200)
plt.show()


**Interpretations (combined from all four members):**
- **Panel 1 (IT25100781 / IT25200148):** `no_tumor` has visibly fewer samples than the three
  tumor classes — this is exactly the imbalance the class weights in Section 8 correct for.
- **Panel 2 (IT25100781):** the four one-hot columns show uniform negative correlation
  (≈ −0.33), confirming the classes are mutually exclusive with no label overlap.
- **Panel 3 (IT25100781):** exactly one active unit per row confirms no ambiguous or
  unencoded targets reach the network.
- **Panel 4 (IT25200148):** pixel intensities are cleanly bounded to `[0, 1]` after Min-Max
  scaling, with no outlier values remaining.
- **Panel 5 (IT25200148):** classes differ somewhat in average brightness, reinforcing why
  normalization (rather than relying on raw scanner intensities) matters before training.
- **Panel 6 (IT25101560):** even in raw pixel space, the four classes show partial separation
  under PCA — a useful sanity check that the label encoding lines up with real visual structure
  before any model is trained.

## Section 10 — On-the-Fly Augmentation Pipeline
**Owner: IT25101560** (adapted to read from the DataFrame split built in Section 4)

IT25101560's original notebook used `ImageDataGenerator.flow_from_directory`, which assumes the
Kaggle folder layout *is* the train/validation split. Since Section 4 now produces its own
leakage-safe, duplicate-aware split as DataFrames, `flow_from_dataframe` is used instead so the
generator reads exactly `fit_train_df` / `val_df` / `test_df` — the augmentation pipeline is
guaranteed to respect the same split as every other section in this notebook, instead of
silently re-deriving a different one from disk.

In [ ]:
# =====================================================================
# SECTION 10: AUGMENTATION GENERATORS — IT25101560
# Requires TensorFlow. Uses flow_from_dataframe so the generator reads the
# exact fit_train_df / val_df / test_df split from Section 4.
# =====================================================================
if TENSORFLOW_AVAILABLE:
    BATCH_SIZE = 32

    # Training generator: rescale + the same augmentations IT25101560 used
    # (rotation, zoom, flip, brightness) to reduce overfitting on a modest MRI dataset.
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255.0,
        rotation_range=20,
        zoom_range=0.15,
        horizontal_flip=True,
        brightness_range=[0.8, 1.2],
    )
    # Validation/test generators: rescale only — no augmentation, so evaluation
    # reflects genuine model performance rather than augmented variants.
    eval_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

    train_generator = train_datagen.flow_from_dataframe(
        dataframe=fit_train_df, x_col="path", y_col="label",
        target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        class_mode="categorical", classes=CLASS_NAMES, shuffle=True, seed=SEED,
    )
    val_generator = eval_datagen.flow_from_dataframe(
        dataframe=val_df, x_col="path", y_col="label",
        target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        class_mode="categorical", classes=CLASS_NAMES, shuffle=False,
    )
    test_generator = eval_datagen.flow_from_dataframe(
        dataframe=test_df, x_col="path", y_col="label",
        target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        class_mode="categorical", classes=CLASS_NAMES, shuffle=False,
    )
    print("Generators ready. Class index mapping:", train_generator.class_indices)
else:
    print("Skipped: install TensorFlow to build the augmentation generators.")


## Section 11 — Pipeline → Model Integration Check
**Owners: IT25100781 (ANN baseline) + IT25101560 (VGG16 transfer learning)**

This is the payoff of the combined pipeline: both members' model architectures are built here
and checked against the *actual* shapes the pipeline produces, rather than shapes assumed in
isolation. The ANN baseline consumes the flattened, normalized array (`X_train`/`y_train`) from
Sections 6–7; the VGG16 head consumes the augmentation generators from Section 10 and is
trained with the class weights from Section 8.

In [ ]:
# =====================================================================
# SECTION 11a: ANN BASELINE — IT25100781
# Flattened grayscale input (IT25100781's original design for a lightweight
# baseline, distinct from the full-RGB VGG16 model below), Softmax output
# sized to match the one-hot targets produced in Section 7.
# =====================================================================
if TENSORFLOW_AVAILABLE:
    # Convert the already-normalized RGB batch to grayscale and flatten, per
    # IT25100781's baseline design (a simple dense network, not a CNN).
    X_train_gray_flat = X_train.mean(axis=-1).reshape(len(X_train), -1)
    print("ANN input shape:", X_train_gray_flat.shape, "| target shape:", y_train.shape)

    ann_baseline = Sequential([
        Input(shape=(IMG_SIZE * IMG_SIZE,)),
        Dense(128, activation="relu"),
        Dropout(0.5),
        Dense(len(CLASS_NAMES), activation="softmax"),   # matches the one-hot width from Section 7
    ])
    ann_baseline.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    ann_baseline.summary()
    assert ann_baseline.input_shape[1] == X_train_gray_flat.shape[1], "ANN input size mismatch!"
    assert ann_baseline.output_shape[1] == y_train.shape[1], "ANN output size mismatch with one-hot targets!"
    print("Shape check passed: ANN baseline matches the pipeline's flattened input and one-hot output.")
else:
    print("Skipped: install TensorFlow to build the ANN baseline.")


In [ ]:
# =====================================================================
# SECTION 11b: VGG16 TRANSFER LEARNING — IT25101560
# Feature extractor frozen (Phase 1), custom classification head sized to
# CLASS_NAMES, trained using the augmentation generators (Section 10) and
# the balanced class weights (Section 8).
# =====================================================================
if TENSORFLOW_AVAILABLE:
    base_model = VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base_model.trainable = False   # Phase 1: train only the new head

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.5)(x)
    outputs = Dense(len(CLASS_NAMES), activation="softmax")(x)

    vgg16_model = Model(inputs=base_model.input, outputs=outputs, name="VGG16_Phase1")
    vgg16_model.compile(optimizer=Adam(learning_rate=1e-3),
                         loss="categorical_crossentropy", metrics=["accuracy"])
    vgg16_model.summary()

    assert vgg16_model.output_shape[1] == len(CLASS_NAMES)
    print("Shape check passed: VGG16 head output matches the number of encoded classes.")

    # --- To actually train (uncomment when ready — omitted here to keep this a
    #      preprocessing-pipeline notebook rather than a full training run): ---
    # callbacks = [
    #     EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    #     ModelCheckpoint(str(OUTPUT_DIR / "best_vgg16_phase1.keras"), monitor="val_loss", save_best_only=True),
    # ]
    # history = vgg16_model.fit(train_generator, validation_data=val_generator, epochs=20,
    #                            class_weight=class_weights, callbacks=callbacks)
else:
    print("Skipped: install TensorFlow to build the VGG16 transfer-learning model.")


## Section 12 — Summary: Contribution Map & Viva Talking Points

| Stage | Technique | Contributor(s) | Key output |
|---|---|---|---|
| 1. Data ingestion | Merge Training/+Testing/ into one DataFrame | IT25101873 | `raw_df` |
| 2. Cleaning | PIL integrity check + cv2 outlier check | IT25200148, IT25101560 | `clean_df` |
| 3. Duplicate detection | SHA-256 hashing, cross-label guard | IT25101873 | `clean_df['sha256']` |
| 4. Splitting | Duplicate-aware stratified 80/20 then 85/15 | IT25101873 | `fit_train_df`, `val_df`, `test_df` |
| 5. Loading/resizing | 224×224, BGR→RGB | IT25200148, IT25101560 | `X_train_raw`, `X_val_raw`, `X_test_raw` |
| 6. Normalization | Min-Max [0,1] + ImageNet standardization | IT25200148 | `X_train`, `X_val`, `X_test` |
| 7. Label encoding | LabelEncoder + OneHotEncoder | IT25100781 | `y_train`, `y_val`, `y_test` |
| 8. Class weighting | `compute_class_weight` (train split only) | IT25101560 | `class_weights` |
| 9. EDA | Distribution, correlation, sparsity, PCA, boxplots | All 4 members | `results/eda_visualizations/` |
| 10. Augmentation | `ImageDataGenerator.flow_from_dataframe` | IT25101560 | `train_generator`, `val_generator` |
| 11. Model integration | ANN baseline + VGG16 head, shape-checked | IT25100781, IT25101560 | `ann_baseline`, `vgg16_model` |

**How this demonstrates collaboration, not just concatenation:**
- Section 4's split DataFrames are the single source of truth every later section reads from —
  no section re-derives its own train/test split from disk.
- Two independent corrupt/outlier checks (Section 2) and two independent encoding approaches
  (Section 7) were reconciled into one shared implementation rather than kept as duplicate code.
- Class weights (Section 8) are computed strictly from the training split so no validation/test
  information leaks into model training.
- Both models built in Section 11 are checked with explicit `assert` statements against the
  pipeline's actual output shapes, not shapes assumed independently by each member.